# Rust 118: Redis

Chapter 18, and chapter 17's client discipline gets a second home. Redis is an **in-memory key-value store with a text protocol**, and the `redis` crate speaks that protocol over async connections — `MultiplexedConnection` clones freely, every command is a round-trip, and expiry is part of the data model, not a cleanup job. The domain is still the trading desk: quotes cached with TTLs (§3, §9), positions as hashes (§4), job queues as lists (§5), watchlists as sets (§6), batched writes as pipelines (§6), and trade events fan-out over pub/sub (§7). Every command in this chapter runs against a **local Redis server built inside the notebook** (§0) — a ~200-line RESP2 implementation on a std `TcpListener` — so every cell is deterministic, offline, and honest: the protocol on the wire is the real Redis protocol, and the behaviors shown are the ones the crate actually produces against it.

Everything here is a **runnable example**. Read the markdown, run the cell below it, and change nothing — the practice twin of this notebook is `18_redis_p.ipynb`.

## Who this is written for

You are assumed to be a competent programmer arriving from Python and C++. `redis-py` and hiredis are old friends; C++ taught you that a socket protocol is just bytes with opinions. What is new in Rust: commands are **methods on a trait** (`AsyncCommands`) whose return types you declare at the call site — the same decode-into-your-type move as §17's `json::<T>()`, but for every command; **the connection is a cheap `Clone`** and one multiplexed socket serves the whole task; **pub/sub is a different connection** with a different grammar — subscribe mode changes what commands mean and what frames arrive; and serialization is **yours to choose** — Redis stores bytes, so §8 puts `serde_json` strings in values and decodes on the way out. The default error story is strict: a server rejection (`unknown command`, wrong type) and a decode mismatch both arrive as one `RedisError`, and telling them apart is §10's skill.

## How to read this notebook

- Markdown cells explain; code cells demonstrate. **Run each code cell as you reach it.** Cells are independent — no sandbox, no files touched, so restart-and-run-from-the-top always works.
- Code that intentionally *fails* is shown as a comment, together with the diagnostic you would get. §10 is the catalogue.
- The running domain is the trading desk: cached quotes, position hashes, price-alert channels. Every API here earns its place on that domain.
- One honesty note runs through the chapter: **the server is local and small**. §0.2 implements the RESP2 wire protocol — the same protocol a real Redis speaks — on an ephemeral port; nothing leaves your machine, no test depends on an external service being up, and the few places a full Redis would behave differently (persistence, clustering, eviction) are marked in §0.2's commentary.


### Contents

- [0. Notebook Conventions: A Local Redis](#0-notebook-conventions-a-local-redis)
- [1. Connecting: Client and MultiplexedConnection](#1-connecting-client-and-multiplexedconnection)
- [2. Strings: GET, SET, INCR](#2-strings-get-set-incr)
- [3. Expiry: EXPIRE, TTL, PERSIST](#3-expiry-expire-ttl-persist)
- [4. Hashes: Fields Within a Key](#4-hashes-fields-within-a-key)
- [5. Lists and Sets](#5-lists-and-sets)
- [6. Pipelines and Transactions](#6-pipelines-and-transactions)
- [7. Pub/Sub](#7-pubsub)
- [8. Values with serde](#8-values-with-serde)
- [9. Cache-Aside: Chapter 17 as a Client](#9-cache-aside-chapter-17-as-a-client)
- [10. Reading the Compiler: E-Codes](#10-reading-the-compiler-e-codes)
- [11. Mental Model: The Receipts](#11-mental-model-the-receipts)
- [12. Cheat Sheet](#12-cheat-sheet)
- [13. Review](#13-review)

If you are returning to this chapter later, §12 is the one-page summary, §10 the diagnostics catalogue, and §9 the pattern the capstone's cache layer is built from.


## 0. Notebook Conventions: A Local Redis

This notebook runs on the **evcxr** kernel, which wraps each cell into a generated program and compiles it incrementally. The standing rules from chapter 16 all hold: no `fn main`, state persists across cells, items are order-independent, one namespace per notebook, first cell that needs a name does the `use`. Four chapter-specific conventions:

- **A cell that wants async creates its runtime and lets it drop** (`Runtime::new()` + `block_on`) — the manual form, because the kernel owns `main`. Every section that talks to the desk does this.
- **The server is local.** §0.2 builds a RESP2 server on `127.0.0.1:0` — the OS picks a free port — and binds its URL for the rest of the notebook. §7's subscribers connect to the same desk on their own connections.
- **Items live before the cell that uses them.** The server's helper `fn`s are items, so any cell could call them; the `desk_url` binding is a *statement* made once in §0.2 and read everywhere after.
- **A TTL is part of the data.** `SET` without options lives forever; `EXPIRE` arms a countdown that erases the key *and everything attached to it* — lists, hashes, sets all die with their key (§3).

### The crates

`redis` (features `tokio-comp` + `serde`) is the client: `tokio-comp` is what makes `get_multiplexed_tokio_connection` and `query_async` exist — §1's entire section is behind a feature flag, chapter 15's lesson again. `serde` + `serde_json` own the value boundary (§8); `tokio` supplies the runtime each cell builds; `futures-util` supplies the `StreamExt` trait §7's message stream needs; and `reqwest` returns from chapter 17 for one cameo — §9's cache-aside origin is an HTTP call.


In [ ]:
// 0.1 The chapter's crate cell: run once per kernel session, features included.
:dep redis = { version = "0.27", default-features = false, features = ["tokio-comp", "serde"] }
:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = "1"
:dep tokio = { version = "1", features = ["full"] }
:dep futures-util = { version = "0.3", default-features = false, features = ["sink", "std"] }
:dep reqwest = { version = "0.12", default-features = false, features = ["json", "charset"] }

// (without "tokio-comp" the async connection types do not exist; the `serde`
// feature pulls serde itself in for the value-conversion impls we use in §8;
// reqwest is chapter 17's client, back for §9's cache-aside origin fetch)
println!("crates loaded: redis, serde, serde_json, tokio, futures-util, reqwest");


### The local desk

A Redis client talks **RESP2**: requests are arrays of bulk strings (`$6\r\nSELECT\r\n`...), replies are one of five prefixed types — `+` simple string, `-` error, `:` integer, `$` bulk string (`$-1` = null), `*` array — **every one CRLF-terminated**. §0.2 implements exactly that grammar plus the command table the chapter uses: strings, expiry, hashes, lists, sets, `PUBLISH`/`SUBSCRIBE`, and `MULTI`/`EXEC` for §6's transactions. Three fidelity notes: the handshake a real Redis gets (`CLIENT SETINFO`, `SELECT`, and on newer servers `HELLO`) is answered tolerantly here — the crate ignores those replies the way a real server's are ignored; expiry is checked *lazily* on access, like real Redis's active-expiry cycle; and there is no persistence or eviction — a teaching desk, not a database.


In [ ]:
// 0.2 The local Redis desk: a raw std TcpListener speaking RESP2.
use std::collections::HashMap;
use std::io::{Read, Write};
use std::sync::{Arc, Mutex};

fn bulk(s: &str) -> String { format!("${}\r\n{}\r\n", s.len(), s) }

fn arr(items: &[String]) -> String {
    let mut out = format!("*{}\r\n", items.len());
    // EVERY element needs its own CRLF: a count element like ":1" without one
    // leaves the frame unterminated and the client's parser waits forever.
    for i in items { out.push_str(&crlf(i.clone())); }
    out
}

// EVERY reply type is CRLF-terminated; the bulk()/arr() builders already end
// with one, and this normalizes the bare "+OK" / ":N" / "-ERR ..." arms too.
fn crlf(mut s: String) -> String {
    if !s.ends_with("\r\n") { s.push_str("\r\n"); }
    s
}

fn find(hay: &[u8], from: usize, needle: &[u8]) -> Option<usize> {
    if needle.len() > hay.len() || from > hay.len() - needle.len() { return None; }
    (from..=hay.len() - needle.len()).find(|&i| &hay[i..i + needle.len()] == needle)
}

fn now_ms() -> u128 {
    std::time::SystemTime::now().duration_since(std::time::UNIX_EPOCH).unwrap().as_millis()
}

#[derive(Default)]
struct Db {
    kv: HashMap<String, String>,
    expiry_ms: HashMap<String, u128>,
    lists: HashMap<String, Vec<String>>,
    sets: HashMap<String, Vec<String>>,
    hashes: HashMap<String, Vec<(String, String)>>, // insertion-ordered fields
    subs: HashMap<String, Vec<usize>>,              // channel -> subscriber conn ids
    psubs: HashMap<String, Vec<usize>>,             // pattern -> subscriber conn ids
}

type SharedDb = Arc<Mutex<Db>>;
type SubTxMap = Arc<Mutex<HashMap<usize, Vec<std::sync::mpsc::Sender<(String, String)>>>>>;

fn alive(d: &Db, key: &str) -> bool {
    let exists = d.kv.contains_key(key) || d.lists.contains_key(key)
        || d.sets.contains_key(key) || d.hashes.contains_key(key);
    exists && d.expiry_ms.get(key).map_or(true, |&t| t > now_ms())
}

/// Expired keys are swept lazily, on access — like real Redis's active cycle.
fn sweep(d: &mut Db) {
    let dead: Vec<String> = d.expiry_ms.iter()
        .filter(|e| *e.1 <= now_ms()).map(|e| e.0.clone()).collect();
    for k in dead {
        d.kv.remove(&k); d.expiry_ms.remove(&k);
        d.lists.remove(&k); d.sets.remove(&k); d.hashes.remove(&k);
    }
}

fn arg(cmd: &[Vec<u8>], i: usize) -> String {
    cmd.get(i).map(|b| String::from_utf8_lossy(b).into_owned()).unwrap_or_default()
}

/// Run one data command against the db. The caller already holds the lock.
fn dispatch(d: &mut Db, subs_tx: &SubTxMap, cmd: &[Vec<u8>]) -> String {
    sweep(d);
    let name = String::from_utf8_lossy(&cmd[0]).to_uppercase();
    match name.as_str() {
        "PING" => "+PONG".to_string(),
        "ECHO" => bulk(&arg(cmd, 1)),
        "SET" | "SETEX" => {
            // the crate's set_ex() sends SETEX (legacy fused write-and-arm)
            let k = arg(cmd, 1);
            if name == "SETEX" {
                let secs: u128 = arg(cmd, 2).parse().unwrap_or(0);
                d.expiry_ms.insert(k.clone(), now_ms() + secs * 1000);
                d.kv.insert(k, arg(cmd, 3));
                return "+OK".to_string();
            }
            let mut ttl = None;
            let mut i = 3;
            while i < cmd.len() {
                let o = arg(cmd, i).to_uppercase();
                if o == "EX" { ttl = Some(now_ms() + arg(cmd, i + 1).parse::<u128>().unwrap_or(0) * 1000); i += 2; }
                else if o == "PX" { ttl = Some(now_ms() + arg(cmd, i + 1).parse::<u128>().unwrap_or(0)); i += 2; }
                else { i += 1; }
            }
            if let Some(t) = ttl { d.expiry_ms.insert(k.clone(), t); }
            d.kv.insert(k, arg(cmd, 2));
            "+OK".to_string()
        }
        "GET" => {
            let k = arg(cmd, 1);
            if alive(d, &k) { bulk(&d.kv[&k]) } else { "$-1\r\n".to_string() }
        }
        "DEL" => {
            let mut n = 0;
            for i in 1..cmd.len() {
                let k = arg(cmd, i);
                let mut gone = d.kv.remove(&k).is_some();
                gone |= d.lists.remove(&k).is_some();
                gone |= d.sets.remove(&k).is_some();
                gone |= d.hashes.remove(&k).is_some();
                d.expiry_ms.remove(&k);
                if gone { n += 1; }
            }
            format!(":{}", n)
        }
        "EXISTS" => format!(":{}", if alive(d, &arg(cmd, 1)) { 1 } else { 0 }),
        "TTL" | "PTTL" => {
            let k = arg(cmd, 1);
            if !alive(d, &k) {
                let lingering = d.kv.contains_key(&k) || d.lists.contains_key(&k)
                    || d.sets.contains_key(&k) || d.hashes.contains_key(&k);
                if lingering { ":-1".to_string() } else { ":-2".to_string() } // expired-but-unswept vs gone
            } else if let Some(&t) = d.expiry_ms.get(&k) {
                let ms = t.saturating_sub(now_ms());
                if name == "PTTL" { format!(":{}", ms) } else { format!(":{}", (ms + 999) / 1000) }
            } else { ":-1".to_string() }
        }
        "EXPIRE" | "PEXPIRE" => {
            let k = arg(cmd, 1);
            let mult: u128 = if name == "EXPIRE" { 1000 } else { 1 };
            let amount: u128 = arg(cmd, 2).parse().unwrap_or(0);
            if alive(d, &k) { d.expiry_ms.insert(k, now_ms() + amount * mult); ":1".to_string() } else { ":0".to_string() }
        }
        "PERSIST" => {
            let k = arg(cmd, 1);
            if alive(d, &k) && d.expiry_ms.remove(&k).is_some() { ":1".to_string() } else { ":0".to_string() }
        }
        "INCR" | "INCRBY" => {
            let k = arg(cmd, 1);
            let delta: i64 = if name == "INCR" { 1 } else { arg(cmd, 2).parse().unwrap_or(0) };
            let v: i64 = d.kv.get(&k).and_then(|s| s.parse().ok()).unwrap_or(0);
            let nv = v + delta;
            d.kv.insert(k, nv.to_string());
            format!(":{}", nv)
        }
        "HSET" => {
            let k = arg(cmd, 1);
            let mut added = 0i64;
            let e = d.hashes.entry(k).or_default();
            let mut a = 2;
            while a + 1 < cmd.len() {
                let f = arg(cmd, a);
                let v = arg(cmd, a + 1);
                if let Some(p) = e.iter_mut().find(|(pf, _)| *pf == f) { p.1 = v; }
                else { e.push((f, v)); added += 1; }
                a += 2;
            }
            format!(":{}", added)
        }
        "HMSET" => { // deprecated variant: +OK, no count
            let k = arg(cmd, 1);
            let e = d.hashes.entry(k).or_default();
            let mut a = 2;
            while a + 1 < cmd.len() {
                let f = arg(cmd, a);
                let v = arg(cmd, a + 1);
                if let Some(p) = e.iter_mut().find(|(pf, _)| *pf == f) { p.1 = v; }
                else { e.push((f, v)); }
                a += 2;
            }
            "+OK".to_string()
        }
        "HGET" => {
            let k = arg(cmd, 1);
            let f = arg(cmd, 2);
            match d.hashes.get(&k).and_then(|h| h.iter().find(|(pf, _)| pf == &f)) {
                Some((_, v)) => bulk(v),
                None => "$-1\r\n".to_string(),
            }
        }
        "HGETALL" => match d.hashes.get(&arg(cmd, 1)) {
            Some(h) => {
                let mut items = Vec::new();
                for (f, v) in h { items.push(bulk(f)); items.push(bulk(v)); }
                arr(&items)
            }
            None => "*0\r\n".to_string(),
        },
        "HDEL" => {
            let k = arg(cmd, 1);
            let f = arg(cmd, 2);
            match d.hashes.get_mut(&k) {
                Some(h) => { let before = h.len(); h.retain(|(pf, _)| pf != &f); format!(":{}", (before - h.len()) as i64) }
                None => ":0".to_string(),
            }
        }
        "LPUSH" => {
            let k = arg(cmd, 1);
            let e = d.lists.entry(k).or_default();
            for i in 2..cmd.len() { e.insert(0, arg(cmd, i)); } // last arg becomes head
            format!(":{}", e.len())
        }
        "RPUSH" => {
            let k = arg(cmd, 1);
            let e = d.lists.entry(k).or_default();
            for i in 2..cmd.len() { e.push(arg(cmd, i)); }
            format!(":{}", e.len())
        }
        "LRANGE" => {
            let k = arg(cmd, 1);
            let start: i64 = arg(cmd, 2).parse().unwrap_or(0);
            let stop: i64 = arg(cmd, 3).parse().unwrap_or(-1);
            match d.lists.get(&k) {
                Some(v) => {
                    let n = v.len() as i64;
                    let s = if start < 0 { (n + start).max(0) } else { start.min(n) };
                    let e = if stop < 0 { n + stop + 1 } else { (stop + 1).min(n) };
                    let e = e.max(s);
                    arr(&v[s as usize..e as usize].iter().map(|x| bulk(x)).collect::<Vec<_>>())
                }
                None => "*0\r\n".to_string(),
            }
        }
        "LLEN" => format!(":{}", d.lists.get(&arg(cmd, 1)).map_or(0, |v| v.len())),
        "SADD" => {
            let k = arg(cmd, 1);
            let mut added = 0;
            let e = d.sets.entry(k).or_default();
            for i in 2..cmd.len() {
                let v = arg(cmd, i);
                if !e.contains(&v) { e.push(v); added += 1; }
            }
            format!(":{}", added)
        }
        "SREM" => {
            let k = arg(cmd, 1);
            let mut removed = 0;
            if let Some(e) = d.sets.get_mut(&k) {
                for i in 2..cmd.len() {
                    let v = arg(cmd, i);
                    if let Some(pos) = e.iter().position(|x| *x == v) { e.remove(pos); removed += 1; }
                }
            }
            format!(":{}", removed)
        }
        "SMEMBERS" => {
            let items: Vec<String> = d.sets.get(&arg(cmd, 1))
                .map(|v| v.iter().map(|x| bulk(x)).collect()).unwrap_or_default();
            arr(&items)
        }
        "SCARD" => format!(":{}", d.sets.get(&arg(cmd, 1)).map_or(0, |v| v.len())),
        "SISMEMBER" => {
            let has = d.sets.get(&arg(cmd, 1)).map_or(false, |s| s.contains(&arg(cmd, 2)));
            format!(":{}", if has { 1 } else { 0 })
        }
        "PUBLISH" => {
            let ch = arg(cmd, 1);
            let payload = arg(cmd, 2);
            let mut count = 0;
            let mut targets: Vec<usize> = d.subs.get(&ch).cloned().unwrap_or_default();
            for (pat, ids) in d.psubs.iter() {                    // pattern subs: trailing *
                if pat.strip_suffix('*').map_or(false, |p| ch.starts_with(p)) {
                    targets.extend(ids.iter().copied());
                }
            }
            let st = subs_tx.lock().unwrap();
            for id in targets {
                if let Some(txs) = st.get(&id) {
                    for t in txs { t.send((ch.clone(), payload.clone())).ok(); count += 1; }
                }
            }
            format!(":{}", count)
        }
        // handshake tolerance: the crate (and real servers) expect these to exist
        "CLIENT" | "SELECT" | "AUTH" => "+OK".to_string(),
        "HELLO" => "-ERR unknown command 'HELLO'".to_string(), // refuse RESP3; the crate stays on RESP2
        "CONFIG" | "COMMAND" => "*0\r\n".to_string(),
        _ => format!("-ERR unknown command '{}'", name),
    }
}

/// One complete command scanned out of a byte buffer. Partial reads stay in
/// the connection buffer and are never discarded — a read that arrives in two
/// TCP segments still parses as one command.
enum Parsed { None, Skip(#[allow(dead_code)] usize), Cmd(Vec<Vec<u8>>, usize) }

fn parse_one(buf: &[u8]) -> Parsed {
    if buf.is_empty() { return Parsed::None; }
    if buf[0] == b'*' {
        let hdr = match find(buf, 0, b"\r\n") { Some(i) => i, None => return Parsed::None };
        let n: usize = match std::str::from_utf8(&buf[1..hdr]).ok().and_then(|s| s.parse().ok()) {
            Some(n) => n, None => return Parsed::Skip(hdr + 2),
        };
        let mut p = hdr + 2;
        let mut args = Vec::new();
        for _ in 0..n {
            if p >= buf.len() { return Parsed::None; }
            if buf[p] != b'$' { return Parsed::Skip(p); }
            let len_end = match find(buf, p, b"\r\n") { Some(i) => i, None => return Parsed::None };
            let len: usize = match std::str::from_utf8(&buf[p + 1..len_end]).ok().and_then(|s| s.parse().ok()) {
                Some(l) => l, None => return Parsed::Skip(len_end + 2),
            };
            p = len_end + 2;
            if p + len + 2 > buf.len() { return Parsed::None; } // partial bulk: wait
            args.push(buf[p..p + len].to_vec());
            p += len + 2;
        }
        Parsed::Cmd(args, p)
    } else {
        let line_end = match find(buf, 0, b"\n") { Some(i) => i, None => return Parsed::None };
        let line = String::from_utf8_lossy(&buf[..line_end]).trim().to_string();
        if line.is_empty() { return Parsed::Skip(line_end + 1); }
        Parsed::Cmd(line.split_whitespace().map(|s| s.as_bytes().to_vec()).collect(), line_end + 1)
    }
}

/// Start the desk; return its URL. Given: the whole chapter tests against this.
fn start_redis_desk() -> String {
    let listener = std::net::TcpListener::bind("127.0.0.1:0").unwrap();
    let port = listener.local_addr().unwrap().port();
    let db: SharedDb = Arc::new(Mutex::new(Db::default()));
    let subs_tx: SubTxMap = Arc::new(Mutex::new(HashMap::new()));
    let next_id = Arc::new(Mutex::new(0usize));
    std::thread::spawn(move || {
        for stream in listener.incoming().flatten() {
            let db = Arc::clone(&db);
            let subs_tx = Arc::clone(&subs_tx);
            let next_id = Arc::clone(&next_id);
            std::thread::spawn(move || {
                let mut r = stream.try_clone().unwrap();
                let mut w = stream;
                // bounded reads so the pub/sub fanout drain gets scheduled
                r.set_read_timeout(Some(std::time::Duration::from_millis(10))).ok();
                let mut my_id: Option<usize> = None;
                let (tx, rx) = std::sync::mpsc::channel::<(String, String)>();
                let mut buf: Vec<u8> = Vec::new();
                let mut tmp = [0u8; 8192];
                let mut queue: Option<Vec<Vec<Vec<u8>>>> = None; // MULTI/EXEC
                loop {
                    // fanout first: queued publishes become RESP push frames
                    loop {
                        match rx.try_recv() {
                            Ok((ch, payload)) => {
                                let frame = arr(&[bulk("message"), bulk(&ch), bulk(&payload)]);
                                w.write_all(frame.as_bytes()).ok();
                                w.flush().ok();
                            }
                            Err(_) => break,
                        }
                    }
                    match r.read(&mut tmp) {
                        Ok(0) => break,
                        Ok(n) => buf.extend_from_slice(&tmp[..n]),
                        Err(ref e) if e.kind() == std::io::ErrorKind::WouldBlock
                            || e.kind() == std::io::ErrorKind::TimedOut => continue,
                        Err(_) => break,
                    }
                    while let Parsed::Cmd(cmd, used) = parse_one(&buf) {
                        buf.drain(..used);
                        if cmd.is_empty() { continue; }
                        let name = String::from_utf8_lossy(&cmd[0]).to_uppercase();
                        // ---- subscribe-mode commands (connection-scoped) ----
                        if name == "SUBSCRIBE" || name == "PSUBSCRIBE" {
                            if my_id.is_none() {
                                let mut g = next_id.lock().unwrap();
                                my_id = Some(*g);
                                *g += 1;
                            }
                            let id = my_id.unwrap();
                            let mut names = Vec::new();
                            {
                                let mut st = subs_tx.lock().unwrap();
                                let mut d = db.lock().unwrap();
                                for a in 1..cmd.len() {
                                    let ch = arg(&cmd, a);
                                    names.push(ch.clone());
                                    st.entry(id).or_default().push(tx.clone());
                                    if name == "SUBSCRIBE" { d.subs.entry(ch).or_default().push(id); }
                                    else { d.psubs.entry(ch).or_default().push(id); }
                                }
                            }
                            let mut confirm = String::new();
                            for (i, ch) in names.iter().enumerate() {
                                // all three elements bulk strings; the count is an int
                                confirm.push_str(&arr(&[bulk(&name.to_lowercase()), bulk(ch), format!(":{}", i + 1)]));
                            }
                            w.write_all(confirm.as_bytes()).ok();
                            w.flush().ok();
                            continue;
                        }
                        if name == "UNSUBSCRIBE" || name == "PUNSUBSCRIBE" {
                            let kind = if name == "UNSUBSCRIBE" { "unsubscribe" } else { "punsubscribe" };
                            let frame = arr(&[bulk(kind), bulk(&arg(&cmd, 1)), ":0".to_string()]);
                            w.write_all(frame.as_bytes()).ok();
                            w.flush().ok();
                            continue;
                        }
                        if name == "PING" && my_id.is_some() {
                            let frame = arr(&[bulk("pong"), bulk("")]);
                            w.write_all(frame.as_bytes()).ok();
                            w.flush().ok();
                            continue;
                        }
                        // ---- MULTI/EXEC: queue while open, run on EXEC ----
                        let reply = match name.as_str() {
                            "MULTI" => {
                                if queue.is_none() { queue = Some(Vec::new()); "+OK".to_string() }
                                else { "-ERR MULTI calls can not be nested".to_string() }
                            }
                            "EXEC" => match queue.take() {
                                None => "-ERR EXEC without MULTI".to_string(),
                                Some(q) => {
                                    let mut out = format!("*{}\r\n", q.len());
                                    for c in q {
                                        let mut d = db.lock().unwrap();
                                        out.push_str(&crlf(dispatch(&mut d, &subs_tx, &c)));
                                    }
                                    out
                                }
                            },
                            "DISCARD" => { queue = None; "+OK".to_string() }
                            _ => {
                                if queue.is_some() {
                                    queue.as_mut().unwrap().push(cmd);
                                    "+QUEUED".to_string()
                                } else {
                                    let mut d = db.lock().unwrap();
                                    dispatch(&mut d, &subs_tx, &cmd)
                                }
                            }
                        };
                        w.write_all(crlf(reply).as_bytes()).ok();
                        w.flush().ok();
                    }
                }
            });
        }
    });
    format!("redis://127.0.0.1:{}", port)
}

let desk_url = start_redis_desk(); // every later cell dials this
println!("local redis desk at {}", desk_url);


Worth reading once, like §0.2 of chapter 17: a connection loop accumulates bytes, `parse_one` scans out whole commands (partial reads wait in the buffer — a command split across TCP segments still parses), `dispatch` runs one command against one shared `Db` behind a mutex, and subscriber connections get a channel that PUBLISH fills and the connection drains as push frames. Every subtlety the client hides — framing, CRLFs, subscribe-mode grammar, MULTI/EXEC queueing — is exactly what the client is *for*. The rest of the chapter asserts against this desk, which is why its claims stay true next year.


## 1. Connecting: Client and MultiplexedConnection

**`redis::Client`** parses a URL (`redis://127.0.0.1:6379/`) and hands out connections; it is cheap to keep and holds no socket itself. The async workhorse is **`get_multiplexed_tokio_connection()`**: one socket, one driver task, and a `MultiplexedConnection` you **`clone()` per call site** — the clone is a handle, not a new connection, so a whole task fan-out shares one TCP stream without a pool. Every command is an `async fn` returning `RedisResult<T>` where **you choose `T`** — `String`, `i64`, `Option<String>`, `Vec<String>` — and the crate decodes the reply into it or errors; the `AsyncCommands` trait supplies the typed method forms (`set`, `get`, `hget`, ...), and `redis::cmd("WHATEVER")` sends any verb the trait has not wrapped. The handshake is automatic: the crate sends `CLIENT SETINFO` twice (both replies ignored), `SELECT` only if the URL names a db, and on newer servers would try `HELLO`; the desk answers all of it, which is why §0.2 tolerates those verbs.


In [ ]:
// 1.1 Connect, PING, and the clone-is-a-handle receipt.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let client = redis::Client::open(desk_url.as_str()).unwrap();
    let con = client.get_multiplexed_tokio_connection().await.unwrap();

    // the clone is a handle to the SAME socket: dropping it keeps the
    // connection alive, and both handles serve commands
    let con2 = con.clone();
    drop(con2); // no disconnect: the original still owns the driver

    let pong: String = redis::cmd("PING").query_async(&mut { con.clone() }).await.unwrap();
    assert_eq!(pong, "PONG");
    println!("multiplexed connection up; PING -> {} (clone-then-drop survived)", pong);
});


Three reads on that cell. **The URL is the configuration**: db number, password, and scheme live in `redis://127.0.0.1:PORT/` — no builder chain, because the protocol has no redirect policies or header defaults to tune. **`query_async` is the raw form**: `redis::cmd("PING")` names the verb, `query_async::<String>` names the reply type — every later cell uses the `AsyncCommands` method forms, which compile to exactly this. **The clone receipt matters architecturally**: chapter 16's `mpsc` clones the transmitter and chapter 17 cloned the whole `Client` per task; here the connection itself clones, and the multiplexed driver interleaves the commands — the connection-per-task cost becomes connection-per-process.


## 2. Strings: GET, SET, INCR

The string type is Redis's everything-type: text, integers-as-text, serialized blobs — **bytes under a key, nothing more**. `set(key, value)` writes, `get(key)` reads as `Option<String>` — **`None` is the missing-key answer, not an error**, the same optionality `HashMap::get` taught in chapter 8. `incr(key, delta)` is the atomic read-modify-write: it parses the stored text as an integer, adds, stores, and returns the new value — a counter with no lock and no lost update even under concurrent tasks, because the whole sequence is one command on the server. `del(key)` returns how many keys it removed.


In [ ]:
// 2.1 SET/GET round-trip, the None-for-missing contract, atomic INCRBY, DEL.
use redis::AsyncCommands;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let mut con = { // one handle for the cell; clone() where tasks would fork
        let client = redis::Client::open(desk_url.as_str()).unwrap();
        client.get_multiplexed_tokio_connection().await.unwrap()
    };

    let _: () = con.set("quote:SPIRE", "101.5").await.unwrap();
    let px: Option<String> = con.get("quote:SPIRE").await.unwrap();
    assert_eq!(px.as_deref(), Some("101.5"));

    let missing: Option<String> = con.get("quote:NOPE").await.unwrap();
    assert!(missing.is_none()); // absence is a value, not a failure

    let first: i64 = con.incr("desk:orders", 1).await.unwrap();
    let second: i64 = con.incr("desk:orders", 9).await.unwrap();
    assert_eq!((first, second), (1, 10)); // atomic read-add-store, server-side

    let gone: i64 = con.del("desk:orders").await.unwrap();
    assert_eq!(gone, 1);
    let after: Option<String> = con.get("desk:orders").await.unwrap();
    assert!(after.is_none());
    println!("set/get round-trip; missing -> None; incr 1 then +9 = 1,10; del = 1");
});


Two reads. **The `(): () =` cast on `set`** is the crate's signature telling you the reply is deliberately discarded — `SET` answers `+OK` and the method maps it to `()`; without the type annotation the compiler cannot infer `T` and will say so (§10's E0282 entry). **`incr` sends `INCRBY` on the wire** (probe-verified), with `INCR` as its delta-of-one twin — and notice the desk stores `10` as the *text* `"10"`: there is no integer type, only strings that parse.


## 3. Expiry: EXPIRE, TTL, PERSIST

Expiry is a per-key countdown, not a garbage collector you schedule: **`expire(key, secs)`** arms it (returns `false` for a key that does not exist — arming nothing is `false`, not an error), **`ttl(key)`** reads it back in one of **three integers**: `-2` the key is gone, `-1` the key lives but has no countdown, `n` seconds remaining. **`persist(key)`** disarms. `set_ex` fuses write-and-arm (`SET ... EX` on the wire); `pexpire`/`pttl` are the millisecond twins for sub-second TTLs — §9's cache uses exactly those. Expired keys vanish **with everything attached**: arm a list or a hash and the countdown kills the whole structure.


In [ ]:
// 3.1 The TTL trichotomy: -2 gone, -1 no countdown, n seconds remaining.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let mut con = {
        let client = redis::Client::open(desk_url.as_str()).unwrap();
        client.get_multiplexed_tokio_connection().await.unwrap()
    };

    let _: () = con.set("sess:u7", "ana").await.unwrap();
    let armed: bool = con.expire("sess:u7", 120).await.unwrap();
    assert!(armed);
    let ttl: i64 = con.ttl("sess:u7").await.unwrap();
    assert!((100..=120).contains(&ttl)); // countdown, slightly under the ask

    let no_countdown: i64 = con.ttl("quote:SPIRE").await.unwrap();
    assert_eq!(no_countdown, -1);        // lives forever until DEL

    let armed_missing: bool = con.expire("ghost", 60).await.unwrap();
    assert!(!armed_missing);             // arming nothing is false, not an error
    let gone: i64 = con.ttl("ghost").await.unwrap();
    assert_eq!(gone, -2);
    println!("TTL says: countdown {}s, no-countdown {}, gone {}", ttl, no_countdown, gone);
});


In [ ]:
// 3.2 Sub-second expiry: pexpire + a wall-clock wait, then persist disarms.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let mut con = {
        let client = redis::Client::open(desk_url.as_str()).unwrap();
        client.get_multiplexed_tokio_connection().await.unwrap()
    };

    let _: () = con.set("tick:hot", "v").await.unwrap();
    let _: () = con.pexpire("tick:hot", 250).await.unwrap(); // milliseconds
    std::thread::sleep(std::time::Duration::from_millis(400)); // wait it out
    let after: Option<String> = con.get("tick:hot").await.unwrap();
    assert!(after.is_none()); // the server deleted it, not the client
    let ttl_after: i64 = con.ttl("tick:hot").await.unwrap();
    assert_eq!(ttl_after, -2);

    // persist: disarm before the countdown fires
    let _: () = con.set("hold", "keep").await.unwrap();
    let _: bool = con.expire("hold", 60).await.unwrap();
    let disarmed: bool = con.persist("hold").await.unwrap();
    assert!(disarmed);
    let ttl_hold: i64 = con.ttl("hold").await.unwrap();
    assert_eq!(ttl_hold, -1);
    println!("pexpire fired server-side (key -2 after 400ms); persist back to -1");
});


Three reads. **The deletion is server-side**: the client slept *past* the deadline and the GET still observed the key gone — expiry is not a client-side filter, the desk's lazy sweep ran inside the GET. **`bool` is the answer type for commands that can no-op**: `expire` on a missing key and `persist` on a key without a countdown both return `false` — Redis says "nothing happened" as data, and only protocol/type trouble is an `Err`. **`pexpire` sends `PEXPIRE`** (probe-verified — the crate's `expire`/`pexpire` do *not* rewrite each other's verbs), which is why the desk implements both.


## 4. Hashes: Fields Within a Key

A hash is **a small map inside one key**: `hset(key, field, value)` adds or overwrites a field (returning how many *new* fields it created — overwriting is `0`), `hget` reads one field as `Option<String>`, `hgetall` returns the whole map, `hdel` removes a field. The desk keeps fields **insertion-ordered** (a real Redis orders them too), so `hgetall` receipts are readable. The habit this builds: reach for a hash when you catch yourself encoding structure into key names (`user:7:name`, `user:7:age`) — one key `user:7` with fields is one expiry to manage, one `DEL` to undo, one round-trip to fetch whole.


In [ ]:
// 4.1 hset counts NEW fields; hget misses are None; hgetall is a HashMap.
use std::collections::HashMap as StdMap;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let mut con = {
        let client = redis::Client::open(desk_url.as_str()).unwrap();
        client.get_multiplexed_tokio_connection().await.unwrap()
    };

    let n1: i64 = con.hset("pos:u7", "sym", "SPIRE").await.unwrap();
    let n2: i64 = con.hset("pos:u7", "qty", "400").await.unwrap();
    let n3: i64 = con.hset("pos:u7", "qty", "450").await.unwrap(); // overwrite
    assert_eq!((n1, n2, n3), (1, 1, 0)); // the third created nothing

    let sym: Option<String> = con.hget("pos:u7", "sym").await.unwrap();
    let ghost: Option<String> = con.hget("pos:u7", "nope").await.unwrap();
    assert_eq!(sym.as_deref(), Some("SPIRE"));
    assert!(ghost.is_none());

    let pos: StdMap<String, String> = con.hgetall("pos:u7").await.unwrap();
    assert_eq!(pos.len(), 2);
    assert_eq!(pos["sym"], "SPIRE");
    assert_eq!(pos["qty"], "450");

    let removed: i64 = con.hdel("pos:u7", "qty").await.unwrap();
    assert_eq!(removed, 1);
    println!("hset counts: 1,1,0; hget missing -> None; hgetall = 2 fields; hdel = 1");
});


Two reads. **`hgetall` decodes into `HashMap<String, String>`** — your type annotation picks the container the way `json::<Vec<Fill>>()` did in §17.3, and `HashMap<String, String>` is the general answer because every field and value arrives as bulk-string bytes. **`hset_multiple` sends legacy `HMSET`** (probe-verified): same fields-in, but the legacy reply is plain `+OK` with **no added-count** — one of several places the modern command and its deprecated twin differ on the wire. One key, many fields, one TTL: the structure-per-key trade in one paragraph.


## 5. Lists and Sets

Two collection types, two access patterns. **A list is an ordered sequence of strings with cheap ends**: `rpush` appends at the tail, `lpush` inserts at the head — **both return the new length** — `lrange(key, start, stop)` slices with **Python-style negatives** (`0, -1` = everything), `llen` counts. The desk-job queue is the classic shape: producers `rpush`, consumers `lpop`/`rpop` from the other end. **A set is an unordered collection of unique strings**: `sadd` adds members and returns how many were **genuinely new** (re-adding is `0`, not an error), `smembers` returns them all (order is not part of the contract — sort before comparing), `sismember` tests, `scard` counts, `srem` removes.


In [ ]:
// 5.1 Lists: push returns the length, lrange slices with negatives.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let mut con = {
        let client = redis::Client::open(desk_url.as_str()).unwrap();
        client.get_multiplexed_tokio_connection().await.unwrap()
    };

    let a: i64 = con.rpush("desk:jobs", "audit").await.unwrap();
    let b: i64 = con.rpush("desk:jobs", "rebalance").await.unwrap();
    let c: i64 = con.lpush("desk:jobs", "urgent").await.unwrap(); // jumps the queue
    assert_eq!((a, b, c), (1, 2, 3));

    let all: Vec<String> = con.lrange("desk:jobs", 0, -1).await.unwrap();
    assert_eq!(all, vec!["urgent", "audit", "rebalance"]); // head first
    let head: Vec<String> = con.lrange("desk:jobs", 0, 0).await.unwrap();
    assert_eq!(head, vec!["urgent"]);
    let len: i64 = con.llen("desk:jobs").await.unwrap();
    assert_eq!(len, 3);
    println!("rpush,rpush,lpush lengths = 1,2,3; lrange 0..-1 = {:?}", all);
});


In [ ]:
// 5.2 Sets: sadd counts genuinely-new members; membership is O(1)-ish test.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let mut con = {
        let client = redis::Client::open(desk_url.as_str()).unwrap();
        client.get_multiplexed_tokio_connection().await.unwrap()
    };

    let added: i64 = con.sadd("watch:u7", &["SPIRE", "QUARK"][..]).await.unwrap();
    let dup: i64 = con.sadd("watch:u7", "SPIRE").await.unwrap();
    assert_eq!((added, dup), (2, 0)); // the duplicate created nothing

    let mut members: Vec<String> = con.smembers("watch:u7").await.unwrap();
    members.sort(); // order is not part of the contract; sort to compare
    assert_eq!(members, vec!["QUARK", "SPIRE"]);
    let watch: bool = con.sismember("watch:u7", "QUARK").await.unwrap();
    assert!(watch);
    let card: i64 = con.scard("watch:u7").await.unwrap();
    assert_eq!(card, 2);
    let gone: i64 = con.srem("watch:u7", "QUARK").await.unwrap();
    assert_eq!(gone, 1);
    println!("sadd new=2 dup=0; smembers sorted = {:?}; scard = {}", members, card);
});


Two reads. **Lengths and counts as return values**: every collection write (`rpush`, `lpush`, `sadd`, `hset`) tells you the new size or the genuinely-added count, so the client never needs a read-back round-trip to know the state — small thing, but it halves the command count of a naive loop. **`&["a", "b"][..]`** is the slice coerced to `&[&str]` — `ToRedisArgs` accepts slices and the wire carries them as separate bulk strings, which is exactly the multi-argument command shape `SADD key m1 m2`.


## 6. Pipelines and Transactions

Two related answers to the same cost: **every command is a round-trip**, and round-trips are the unit of latency. **A pipeline** (`redis::pipe()`) queues several commands client-side and writes them **in one go**, reading all the replies — batching without any server semantics; each reply lands in its tuple slot. **`.ignore()`** marks a command as reply-discarded — and probe-verified here: an ignored command contributes **no tuple slot** — a three-command pipe with two ignores decodes `(one_value,)`. **`.atomic()`** wraps the queue in **`MULTI` ... `EXEC`** on the wire: the server executes all queued commands *and nothing else may interleave* — a transaction. The desk queues between MULTI and EXEC (each queued command answers `+QUEUED`) and answers EXEC with **one array of per-command replies**, which is what the crate's tuple decode expects.


In [ ]:
// 6.1 A pipeline: three commands, one write, replies decoded positionally.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let mut con = {
        let client = redis::Client::open(desk_url.as_str()).unwrap();
        client.get_multiplexed_tokio_connection().await.unwrap()
    };

    let _: () = con.set("px:SPIRE", "101.5").await.unwrap();
    let _: () = con.sadd("watch:u7", "SPIRE").await.unwrap();

    // GET ignored (we just wrote it), SCARD kept: (i64,) is the WHOLE tuple
    let (card,): (i64,) = redis::pipe()
        .cmd("GET").arg("px:SPIRE").ignore()
        .cmd("SCARD").arg("watch:u7")
        .query_async(&mut con)
        .await
        .unwrap();
    assert_eq!(card, 1);
    println!("pipeline: GET ignored (no slot), SCARD decoded = {}", card);
});


In [ ]:
// 6.2 atomic(): MULTI ... EXEC — the write+read pair no other task can interleave.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let mut con = {
        let client = redis::Client::open(desk_url.as_str()).unwrap();
        client.get_multiplexed_tokio_connection().await.unwrap()
    };

    let (read_back,): (String,) = redis::pipe()
        .atomic()
        .cmd("SET").arg("tx:hold").arg("locked").ignore()
        .cmd("GET").arg("tx:hold")
        .query_async(&mut con)
        .await
        .unwrap();
    assert_eq!(read_back, "locked");
    println!("atomic pipe: SET+GET as one transaction, read back = {}", read_back);
});


Three reads. **`ignore()` moves the type, not the value**: the GET's reply still arrived (the desk computed it) but the crate skipped decoding it into your tuple — position is a *type* decision at the call site, which is why the tuple is `(i64,)` and not `(String, i64)`. **The atomic guarantee is about interleaving, not rollbacks**: if the connection dies mid-EXEC the commands may have run — there is no undo, only "no other client's commands got between ours." **When atomic matters**: read-modify-write pairs (check a set, then add; read a counter, then bump by a computed amount) — §2's `INCR` is the server doing this for you in one command; `atomic()` generalizes it to any command sequence.


## 7. Pub/Sub

**Publish/subscribe is fire-and-forget fan-out**: `PUBLISH channel payload` returns **how many subscribers received it** — zero is a perfectly good answer (no one was listening; the message is gone; there is no history and no replay). A subscriber connection enters **subscribe mode**, where the grammar changes: commands it may send shrink to `SUBSCRIBE`/`UNSUBSCRIBE`/`PING`/etc., and replies arrive as **three-element arrays** — `[kind, channel, count]` for confirmations (`subscribe`, `unsubscribe`, each with the running subscription count) and `[message, channel, payload]` for deliveries. **The crate's client side**: `client.get_async_pubsub()` opens a **dedicated socket** (a multiplexed connection cannot enter subscribe mode — its shared socket is busy serving regular commands), `subscribe(channel)` sends the request, and **`into_on_message()`** consumes the connection into a `Stream<Msg>` — the `while let Some(msg) = stream.next().await` spine from §17.9, delivering server-push frames this time. `msg.get_channel_name()` and `msg.get_payload::<T>()` decode the frame's parts.


In [ ]:
// 7.1 PUBLISH counts receivers: zero before anyone subscribes, one after.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let client = redis::Client::open(desk_url.as_str()).unwrap();
    let mut con = client.get_multiplexed_tokio_connection().await.unwrap();

    let before: i64 = redis::cmd("PUBLISH").arg("fills").arg("SPIRE@101.5")
        .query_async(&mut con).await.unwrap();
    assert_eq!(before, 0); // fire-and-forget into the void

    // dedicated socket; a multiplexed connection cannot subscribe
    let mut pubsub = client.get_async_pubsub().await.unwrap();
    pubsub.subscribe("fills").await.unwrap();
    let mut stream = pubsub.into_on_message(); // consumes the connection
    std::thread::sleep(std::time::Duration::from_millis(150)); // let it register

    let after: i64 = redis::cmd("PUBLISH").arg("fills").arg("QUARK@44.25")
        .query_async(&mut con).await.unwrap();
    assert_eq!(after, 1);

    use futures_util::StreamExt;
    let msg = tokio::time::timeout(std::time::Duration::from_secs(3), stream.next())
        .await
        .expect("message within 3s")
        .expect("stream delivers");
    assert_eq!(msg.get_channel_name(), "fills");
    let payload: String = msg.get_payload().unwrap();
    assert_eq!(payload, "QUARK@44.25");
    println!("publish before subs = 0, with sub = 1; got channel={} payload={}",
        msg.get_channel_name(), payload);
    drop(stream); // dropping the stream drops the subscriber socket
});


Three reads. **The count is the delivery receipt**: `PUBLISH` tells you exactly how many subscribers got the frame — this is how a test knows the message landed, and how production notices "alerts are on but nobody is subscribed." **The sleep is honest**: the SUBSCRIBE request and its confirmation are one round-trip, but the *publisher* must observe the registration, so a beat of wall-clock separates subscribe from publish — chapter 16's "concurrency needs synchronization points" in miniature. **Two connection kinds, two grammars**: the multiplexed socket speaks command/reply; the subscriber socket speaks command/push-stream — `into_on_message()` (which *consumes* the `PubSub`) versus `on_message()` (which borrows it and blocks `unsubscribe` until dropped) is the ownership choice, probed in §10.


## 8. Values with serde

Redis stores **bytes**; what those bytes *mean* is your convention. The chapter's convention is the boundary-struct one from chapter 17: **`serde_json::to_string` into a string value, `serde_json::from_str` on the way out** — typed values in, typed values out, and the cache in §9 is built on it. Derive `Serialize`/`Deserialize` on a plain owned struct (same rules as §17.3), write it with `set`, read it with `get::<Option<String>>` and decode — a decode failure at the edge is your contract catching a wire change, not a Redis error. (The `redis` crate's `serde` feature adds conversion impls for map-like values; for whole-object round-trips the JSON-string convention is the pragmatic default and keeps the value readable with any Redis client.)


In [ ]:
// 8.1 A derived struct through the desk as JSON, typed both ways.
#[derive(serde::Serialize, serde::Deserialize, Debug, PartialEq, Clone)]
struct Quote {
    sym: String,
    px: f64,
    seq: u64,
}

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let mut con = {
        let client = redis::Client::open(desk_url.as_str()).unwrap();
        client.get_multiplexed_tokio_connection().await.unwrap()
    };

    let q = Quote { sym: "SPIRE".into(), px: 101.5, seq: 7 };
    let _: () = con.set("json:quote:SPIRE", serde_json::to_string(&q).unwrap())
        .await.unwrap();

    let raw: Option<String> = con.get("json:quote:SPIRE").await.unwrap();
    let back: Quote = serde_json::from_str(&raw.unwrap()).unwrap();
    assert_eq!(back, q);
    println!("serde round-trip: {:?} -> bytes -> {:?}", q, back);

    // the wire truth: it is just a string value; any client could read it
    let as_text: Option<String> = con.get("json:quote:SPIRE").await.unwrap();
    assert!(as_text.unwrap().starts_with("{\"sym\":\"SPIRE\""));
    println!("stored form is plain JSON text under the key");
});


Two reads. **The value is self-describing**: §2's `"101.5"` needed the caller to know it parses as a number; the JSON blob names its own fields, which is what lets §9's cache be *generic over callers* — anyone with the struct definition decodes, anyone without still sees sensible text. **`PartialEq` on the derive list** is there so the round-trip can be asserted — the same derive-driven testability chapter 15's `rust_decimal` receipts used; `Clone` appears because §9's demo moves the struct into a closure.


## 9. Cache-Aside: Chapter 17 as a Client

The pattern that pays for this whole chapter: **cache-aside**. On a request for data: (1) `GET` the cache key; (2) on a hit, decode and return — the origin never hears about it; (3) on a miss, **fetch from the origin** — here, the §17.2-style local HTTP server — `SET` the value **with a TTL**, return it. The TTL is the correctness argument: stale data self-destructs, so the cache never needs an invalidation protocol, just a freshness budget. Note what the pattern does *not* do: it does not pre-warm, does not write through, does not lock against two concurrent misses (both fetch; both set; last write wins — harmless here). The desk's `pexpire`-precision TTLs make the demonstration honest: a 250ms budget visibly expires *within* the demo.


In [ ]:
// 9.1 The origin: a tiny HTTP server (chapter 17's §0.2 pattern).
use std::io::BufRead;

fn start_origin() -> String {
    let listener = std::net::TcpListener::bind("127.0.0.1:0").unwrap();
    let port = listener.local_addr().unwrap().port();
    std::thread::spawn(move || {
        for stream in listener.incoming().flatten() {
            let mut s = stream;
            let mut reader = std::io::BufReader::new(match s.try_clone() { Ok(r) => r, Err(_) => return });
            let mut line = String::new();
            if reader.read_line(&mut line).is_err() { return; }
            let path = line.split_whitespace().nth(1).unwrap_or("/").to_string();
            loop { // drain headers to the blank line
                let mut h = String::new();
                if reader.read_line(&mut h).is_err() || h.trim().is_empty() { break; }
            }
            let sym = path.trim_start_matches("/quote/").to_uppercase();
            let body = format!("{{\"sym\":\"{}\",\"px\":{}.5,\"seq\":1}}", sym,
                if sym == "SPIRE" { 101 } else { 55 });
            let head = format!(
                "HTTP/1.1 200 OK\r\nContent-Type: application/json\r\nContent-Length: {}\r\nConnection: close\r\n\r\n",
                body.len());
            s.write_all(head.as_bytes()).ok();
            s.write_all(body.as_bytes()).ok();
        }
    });
    format!("http://127.0.0.1:{}", port)
}

let origin_url = start_origin();
println!("origin up at {} (chapter 17 pattern: parse, answer, close)", origin_url);


In [ ]:
// 9.2 cache-aside: GET, miss -> origin fetch -> SET EX, hit -> decode and return.
#[derive(serde::Serialize, serde::Deserialize, Debug, PartialEq, Clone)]
struct CachedQuote {
    sym: String,
    px: f64,
    seq: u64,
}

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let client = redis::Client::open(desk_url.as_str()).unwrap();
    let mut con = client.get_multiplexed_tokio_connection().await.unwrap();
    let http = reqwest::Client::new(); // chapter 17's client, §0.1's cell

    // async fn form: the pattern as one function of (connection, symbol)
    async fn fetch(
        con: &mut redis::aio::MultiplexedConnection,
        http: &reqwest::Client,
        origin: &str,
        sym: &str,
    ) -> (CachedQuote, bool) {
        let key = format!("cache:quote:{}", sym);
        let hit: Option<String> = con.get(&key).await.unwrap();
        if let Some(raw) = hit {
            return (serde_json::from_str::<CachedQuote>(&raw).unwrap(), true);
        }
        let body: String = http.get(format!("{}/quote/{}", origin, sym))
            .send().await.unwrap().text().await.unwrap();
        let q: CachedQuote = serde_json::from_str(&body).unwrap();
        let _: () = con.set_ex(&key, serde_json::to_string(&q).unwrap(), 60u64)
            .await.unwrap(); // SET ... EX: write and arm in one command
        (q, false)
    }

    let (first, was_hit) = fetch(&mut con, &http, &origin_url, "SPIRE").await;
    assert!(!was_hit, "first read is a miss");
    let (second, was_hit2) = fetch(&mut con, &http, &origin_url, "SPIRE").await;
    assert!(was_hit2, "second read hits");
    assert_eq!(first, second);

    let ttl: i64 = con.ttl("cache:quote:SPIRE").await.unwrap();
    assert!((55..=60).contains(&ttl)); // the miss armed the countdown
    println!("miss then hit for {} @ {}; cache TTL = {}s", first.sym, first.px, ttl);
});


In [ ]:
// 9.3 The expiry end of the bargain: short TTL, wait, miss again.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let mut con = {
        let client = redis::Client::open(desk_url.as_str()).unwrap();
        client.get_multiplexed_tokio_connection().await.unwrap()
    };

    let _: () = con.set_ex("cache:quote:QUARK", "{\"sym\":\"QUARK\",\"px\":55.5,\"seq\":1}", 1u64)
        .await.unwrap();
    std::thread::sleep(std::time::Duration::from_millis(1200)); // outlive the budget
    let gone: Option<String> = con.get("cache:quote:QUARK").await.unwrap();
    assert!(gone.is_none()); // stale data self-destructed; next read is a miss
    println!("1s budget expired server-side; the next caller re-fetches from the origin");
});


Three reads. **`set_ex` is the pattern's verb**: `SET key value EX 60` on the wire — the write and the freshness budget in one atomic command, no window where a cached value lives forever because the process died between `set` and `expire`. **`fetch` is an `async fn` taking the connection by `&mut`** — a plain item, not a closure returning an `async move` block, because the closure form fights the borrow checker over the future's lifetime (the `async fn` signature states the borrow once and the compiler believes it). **The hit/miss tuple** (`(CachedQuote, bool)`) is the observability seam: production logs the ratio, the demo asserts it, and both are the same boolean.


## 10. Reading the Compiler: E-Codes

| Diagnostic | The story it tells |
|---|---|
| **E0282** | Type annotations needed — the command's `T` is uninferred: `con.get("k")` without a binding's type; fix by annotating the binding (`let v: Option<String> = ...`) or turbofishing `con.get::<_, Option<String>>("k")`. |
| **E0277** | `MultiplexedConnection: redis::aio::ConnectionLike` not satisfied for a *sync* call — `con.get()` without `.await` in async context, or the `tokio-comp` feature off so the async connection types do not exist. |
| **E0599** | No method `set`/`get`/`hset` on the connection — `AsyncCommands` not imported (`use redis::AsyncCommands;`); or `into_on_message` absent — `futures_util::StreamExt` missing for the stream side. |
| **E0499** | `pubsub` borrowed mutably twice — `let s = pubsub.on_message()` holds the borrow; `unsubscribe` (or another method) cannot take it until `s` drops. The fix is `into_on_message()`, which *consumes*. |
| **E0308** | The command fired but the reply does not fit your type — a `String` where you declared `i64`, or a map where you declared a scalar; RedisError's `TypeError` variant names it at runtime as "Response was of incompatible type". |
| **`RedisError`** | Not a rustc code: two families — **server-signalled** (`An error was signalled by the server - unknown command '...'`, `ResponseError`) and **decode/type** (`TypeError`, `ExtensionError`) — `err.kind()` discriminates, §6's wire knowledge tells you which side produced it. |


In [ ]:
// 10.1 The E0282 of an uninferred command type, demonstrated in a comment.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let mut con = {
        let client = redis::Client::open(desk_url.as_str()).unwrap();
        client.get_multiplexed_tokio_connection().await.unwrap()
    };
    let _: () = con.set("e:demo", "42").await.unwrap();

    // let v = con.get("e:demo").await.unwrap();
    //     ^ error[E0282]: type annotations needed
    //       cannot infer type of the type parameter `T` declared on
    //       the associated function `get`  — the reply type IS your choice

    let v: Option<String> = con.get("e:demo").await.unwrap(); // annotated: compiles
    assert_eq!(v.as_deref(), Some("42"));
    println!("the reply type is chosen at the call site; annotation satisfies E0282");
});


In [ ]:
// 10.2 The two RedisError families, shown as runtime behavior (no panic).
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let mut con = {
        let client = redis::Client::open(desk_url.as_str()).unwrap();
        client.get_multiplexed_tokio_connection().await.unwrap()
    };

    // family 1: the server refused the verb (the desk, like a real Redis,
    // answers -ERR for commands it does not know)
    let bad = redis::cmd("NOSUCHCMD").arg("x").query_async::<String>(&mut con).await;
    match bad {
        Err(e) => println!("server-signalled: kind={:?} {}", e.kind(), e),
        Ok(_) => panic!("desk rejected nothing?"),
    }

    // family 2: the verb worked but the reply does not fit the declared type
    let _: () = con.set("e:str", "not-a-number").await.unwrap();
    let misfit = con.get::<_, Option<i64>>("e:str").await;
    match misfit {
        Err(e) => println!("decode-side: kind={:?} {}", e.kind(), e),
        Ok(_) => unreachable!("i64 cannot parse from that text"),
    }
});


Two reads. **10.1 is §1's "you choose `T`" seen as a compiler behavior**: the same line that fails with E0282 compiles the moment the binding names its type — the annotation is not ceremony, it *is* the decode target. **10.2's two families deserve the discipline**: `err.kind()` is the discriminator (`ResponseError` vs `TypeError`), and knowing which side failed decides the reflex — a server-signalled error means the *verb or data shape* is wrong (fix the call); a type error means the *declared type* is wrong (fix the annotation). The desk's INCR-by-non-number behavior differs from real Redis here (real Redis errors, the desk's parse-or-zero is lenient) — noted so the honesty ledger stays balanced.


## 11. Mental Model: The Receipts

| Claim | Receipt |
|---|---|
| The URL is the configuration; the client is cheap | §1.1: `Client::open(desk_url)`, one `get_multiplexed_tokio_connection` |
| A clone is a handle, not a connection | §1.1: `con.clone()` dropped, PING still answers |
| Absence is a value, not a failure | §2.1: `get` of a missing key → `Ok(None)` |
| Counters are one atomic command | §2.1: `incr` 1 then +9 → 1, 10 |
| TTLs have three answers | §3.1: countdown n, no-countdown −1, gone −2 |
| Expiry is server-side | §3.2: 250ms `pexpire`, client sleeps 400ms, key gone |
| Arming nothing is `false` | §3.1: `expire("ghost", 60)` → `Ok(false)` |
| A hash is one key with fields | §4.1: hset counts new fields (1,1,0), one key, one DEL |
| Push returns the new size | §5.1: rpush,rpush,lpush → 1,2,3 |
| Sets return genuinely-added | §5.2: sadd new=2, dup=0 |
| Pipelines batch; ignore drops the slot | §6.1: GET ignored, `(i64,)` decodes SCARD |
| atomic() = MULTI/EXEC, no interleaving | §6.2: +QUEUED, EXEC answers an array |
| PUBLISH counts receivers | §7.1: 0 before, 1 after subscribe |
| Pub/sub is a dedicated socket and a stream | §7.1: `get_async_pubsub` + `into_on_message` |
| Values are bytes; serde gives them meaning | §8.1: struct → JSON string → same struct |
| Cache-aside = GET, miss→origin→SET EX, hit→decode | §9.2: miss then hit, TTL armed by the miss |
| Staleness self-destructs | §9.3: 1s TTL, sleep, `None` |


## 12. Cheat Sheet

```rust
// ---- connect (once per process; clone the connection freely) ----
let client = redis::Client::open("redis://127.0.0.1:6379/")?;
let con = client.get_multiplexed_tokio_connection().await?;   // tokio-comp feature
let mut c2 = con.clone();                                      // handle, not socket
let pong: String = redis::cmd("PING").query_async(&mut con.clone()).await?;

// ---- strings (use redis::AsyncCommands;) ----
let _: () = con.set("k", "v").await?;                          // +OK mapped to ()
let v: Option<String> = con.get("k").await?;                   // None = absent
let n: i64 = con.incr("hits", 1).await?;                       // INCRBY: atomic add
let gone: i64 = con.del("k").await?;                           // keys removed

// ---- expiry: three TTL answers ----
let armed: bool = con.expire("k", 60).await?;                  // false if absent
let _: () = con.set_ex("k", "v", 60u64).await?;                // SET ... EX, fused
let _: () = con.pexpire("k", 250).await?;                      // milliseconds
let t: i64 = con.ttl("k").await?;      // -2 gone, -1 no countdown, n seconds
let _: bool = con.persist("k").await?;                         // disarm

// ---- hashes, lists, sets ----
let added: i64 = con.hset("user:7", "name", "ana").await?;     // NEW fields only
let one: Option<String> = con.hget("user:7", "name").await?;
let all: std::collections::HashMap<String, String> = con.hgetall("user:7").await?;
let len: i64 = con.rpush("jobs", "a").await?;                  // or lpush (head)
let slice: Vec<String> = con.lrange("jobs", 0, -1).await?;     // negatives ok
let fresh: i64 = con.sadd("tags", &["a", "b"][..]).await?;     // genuinely-new
let members: Vec<String> = con.smembers("tags").await?;        // sort to compare
let has: bool = con.sismember("tags", "a").await?;

// ---- pipelines and transactions ----
let (card,): (i64,) = redis::pipe()
    .cmd("GET").arg("k").ignore()          // ignore() = NO tuple slot
    .cmd("SCARD").arg("tags")
    .query_async(&mut con).await?;
let (v,): (String,) = redis::pipe().atomic()   // MULTI ... EXEC: no interleaving
    .cmd("SET").arg("tx").arg("1").ignore()
    .cmd("GET").arg("tx")
    .query_async(&mut con).await?;

// ---- pub/sub (dedicated socket; into_on_message consumes) ----
let mut pubsub = client.get_async_pubsub().await?;
pubsub.subscribe("alerts").await?;
let mut stream = pubsub.into_on_message();     // Stream<Msg>; on_message borrows
while let Some(msg) = stream.next().await {    // futures_util::StreamExt
    let ch = msg.get_channel_name();
    let payload: String = msg.get_payload()?;  // PUBLISH count = receivers
}

// ---- serde values (the cache-aside core) ----
#[derive(serde::Serialize, serde::Deserialize, PartialEq, Clone)]
struct Quote { sym: String, px: f64, seq: u64 }
let _: () = con.set("q:SPIRE", serde_json::to_string(&q)?).await?;
let back: Quote = serde_json::from_str(&con.get::<_, Option<String>>("q:SPIRE")
    .await?.unwrap())?;
```

Seven load-bearing sentences: **the URL is the configuration and the client hands out cloneable multiplexed connections; every command's reply type is chosen at the call site; absence is `Ok(None)`, not an error; expiry is server-side data with three TTL answers; pipelines batch round-trips and `ignore()` drops the tuple slot while `atomic()` adds MULTI/EXEC; pub/sub runs on a dedicated socket as a push stream whose `PUBLISH` counts receivers; values are bytes and `serde_json` is the convention that makes them typed.**


## 13. Review

Twelve questions, in chapter order. Answers are in the prose above each.

1. **(§0)** What is RESP2's reply alphabet, and what does every reply type end with?
2. **(§1)** What does `con.clone()` actually copy, and what does dropping a clone *not* do?
3. **(§1)** Where do you declare what type a command's reply decodes into, and what fails to compile if you don't?
4. **(§2)** Why is `get` returning `Ok(None)` better than an error for a missing key, and which chapter's container taught the same shape?
5. **(§2)** What does `incr` send on the wire, and why is it immune to the lost-update race two `get`-then-`set` callers have?
6. **(§3)** Name the three `ttl` answers and what each says about the key.
7. **(§3)** Who deletes an expired key — the client that armed it or the server — and where in §0.2 does that happen?
8. **(§4)** What does `hset` count, and why is one hash under `user:7` easier to manage than `user:7:name`-style keys?
9. **(§5)** Which pushes return the new length, which add returns the genuinely-new count, and why does `smembers` order demand sorting before comparison?
10. **(§6)** What does `.ignore()` remove from the pipeline's result, and what guarantee does `.atomic()` add over a plain pipeline?
11. **(§7)** Why does pub/sub need a dedicated connection, what does `PUBLISH`'s return value tell you, and what arrives in a subscriber's push frame?
12. **(§9)** Walk the cache-aside path for a miss, name the fused command that writes-and-arms, and explain why the TTL is the correctness argument.

### What Comes Next

| Next | What it borrows from here |
|---|---|
| **19 — PostgreSQL** | `sqlx` pools are §1's multiplexed connection made relational; §6's transactions survive `await` points verbatim; §9's cache-aside sits in front of real queries. |
| **20 — PyO3** | the escape hatch back to Python: `redis-py` returns strings you parse; the typed-command discipline here is what the extension exports. |
| **21 — Capstone Prep** | §9's cache in front of §17's feed plus §19's storage is the market-data service the capstone assembles; §7's fan-out becomes its event bus. |

Carry forward: **the reply type is yours to declare; absence is a value; expiry is the invalidation protocol; the pattern is GET, miss→origin→SET EX, hit→decode.**
